In [2]:
!pip install langchain
!pip install langchain-community
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf

In [4]:
from google.colab import files

uploaded = files.upload()

Saving tools.py to tools (1).py


In [5]:
import tools

print(tools.__file__)

/content/tools.py


In [7]:
from tools import get_current_time

print(get_current_time())

2026-06-02 09:25:29


In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/d_t4_2_2_technical_documentation_the_for_web_based_tool_final_version.pdf")

documents = loader.load()

print("Pages:", len(documents))

/tmp/ipykernel_2629/1036774463.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages: 22


In [9]:
 !pip install langchain-text-splitters

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

print("Chunks:", len(docs))

Chunks: 62


In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2629/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
    docs,
    embeddings
)

db.save_local("vectorstore")

print("Vector DB Created")

Vector DB Created


In [14]:
def retrieve(query):

    results = db.similarity_search(
        query,
        k=3
    )

    context = "\n".join(
        [r.page_content for r in results]
    )

    return context[:1000]

In [15]:
from tools import get_current_time

In [16]:
print(get_current_time())

2026-06-02 09:29:39


In [18]:
import tools as tools
print(tools.__file__)

/content/tools.py


In [19]:
from tools import get_current_time

print(get_current_time())

2026-06-02 09:30:01


In [20]:
chat_history = []

def rewrite_question(
    question,
    history
):

    if len(history) == 0:
        return question

    previous_question = history[-1]["question"]

    return f"""
Previous Question:
{previous_question}

Current Question:
{question}
"""

In [21]:
def route_question(question):

    if "time" in question.lower():
        return "tool"

    return "rag"

In [22]:
question = input("You: ")

standalone_question = rewrite_question(
    question,
    chat_history
)

route = route_question(question)

if route == "tool":

    answer = get_current_time()

else:

    answer = retrieve(
        standalone_question
    )

print("\nAssistant:")
print(answer)

chat_history.append({
    "question": question,
    "answer": answer
})

You: what are web application 

Assistant:
#1 This document is based on users’ definitions of web-tool requirements.  
#2 It’s extended with the author’s suggestions based on practical experience with 
similar programs and with theoretical principles for software development. 
0.6 SPECIFIC DESIGN CONSIDERATIONS 
#1 Specifics regarding design of the web-based tool are considered by user 
requirements.  
#2 Specifics regarding the type and amount of the users are also considered by 
user requirements.
requirements for the final implementation of the web-based tool. 
#3 The document is open-ended and it shall be modified according to the final 
round of the implementation of the web-tool. No final solution has been 
provided for the final vendor as agreed upon with the investor. Differences 
between this document and the implementation of the web-based tool shall 
result in appropriate revisions of the document. 
#4 In the planning and defining phase of the development-process all types o